# MaritimeBERT Domain-Adaptive Pretraining (DAPT) - Google Colab Runner

This notebook executes the **ModernBERT DAPT Pipeline** (`MaritimeBERT-v1`) entirely on Google Colab's cloud GPU hardware.

--- 

## Step 1: Ensure Files are Available to Colab

### Option A: Mount Google Drive (Recommended for persistent checkpoints & metrics)
Run Cell 1 below to mount Google Drive and navigate to your uploaded project directory.

### Option B: Direct GitHub / Upload
If you uploaded or cloned the project directly into `/content/TSBC-Pipeline`, navigate into your repository root directory.

--- 

## Step 2: Set Colab Runtime to GPU
1. Go to **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or A100/V100 if available).
3. Click **Save**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 1: Mount Google Drive & Set Working Directory
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Adjust this path if your project directory name in Drive is different
    drive_project_path = '/content/drive/MyDrive/TSBC-Pipeline'
    if os.path.exists(drive_project_path):
        os.chdir(drive_project_path)
        print("Successfully changed working directory to:", os.getcwd())
    else:
        print("Current working directory:", os.getcwd())
except Exception as e:
    print("Drive mount skipped or not running in Colab:", e)

## 1. Verify GPU Hardware Runtime

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("GPU Memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
!nvidia-smi

## 2. Install DAPT Pipeline Requirements

In [ ]:
!pip install -r dapt/requirements.txt

## 3. Step 1: Corpus Inspection & Cryptographic SHA-256 Hash Lock

In [ ]:
!python dapt/scripts/inspect_corpus.py --config dapt/configs/dapt.yaml

## 4. Step 2: Prepare Reproducible Dataset Splits & Leakage Diagnostics

In [ ]:
!python dapt/scripts/prepare_dataset.py --config dapt/configs/dapt.yaml

## 5. Step 3: ModernBERT Tokenizer Diagnostics

In [ ]:
!python dapt/scripts/tokenize_corpus.py --config dapt/configs/dapt.yaml

## 6. Step 4: Validate Dataset & Document Sequence Packing Efficiency

In [ ]:
!python dapt/scripts/validate_dataset.py --config dapt/configs/dapt.yaml

## 7. Step 5: Evaluate Untouched ModernBERT Baseline on Held-Out Set

In [ ]:
!python dapt/scripts/evaluate_dapt.py --config dapt/configs/dapt.yaml --is-baseline --eval-split validation

## 8. Step 6: Execute ModernBERT Domain-Adaptive Pretraining (MaritimeBERT-v1) on GPU

In [ ]:
!python dapt/scripts/train_dapt.py --config dapt/configs/dapt.yaml

## 9. Step 7: Evaluate Trained MaritimeBERT-v1 Model

In [ ]:
!python dapt/scripts/evaluate_dapt.py --config dapt/configs/dapt.yaml --eval-split validation

## 10. Step 8: Compare Baseline ModernBERT vs. MaritimeBERT-v1 Metrics

In [ ]:
!python dapt/scripts/compare_runs.py

## 11. Run Unit Test Suite (Optional Verification)

In [ ]:
!pytest dapt/tests/